In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from torch.utils.data import Dataset, DataLoader
import pickle

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("nifty_features.csv")

target = 'target'

# =========================
# FEATURE SELECTION (Paper Idea)
# =========================
features = [
    'log_ret','ret_1','ret_5','ret_15',
    'vol_5','vol_15',
    'body','range','upper_wick','lower_wick','close_pos',
    'dist_ma_5','dist_ma_20',
    'volume_spike','bank_ret','it_ret'
]

X = df[features]
y = df[target]

mi = mutual_info_regression(X, y)
mi_series = pd.Series(mi, index=features)

selected_features = mi_series.sort_values(ascending=False).head(10).index.tolist()
print("Selected Features:", selected_features)

X = df[selected_features].values
y = y.values

# =========================
# SCALE
# =========================
scaler = StandardScaler()
X = scaler.fit_transform(X)

with open("lstm_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("lstm_features.pkl", "wb") as f:
    pickle.dump(selected_features, f)

# =========================
# SEQUENCE
# =========================
SEQ_LEN = 20

def create_sequences(X, y):
    X_seq, y_seq = [], []
    for i in range(len(X) - SEQ_LEN):
        X_seq.append(X[i:i+SEQ_LEN])
        y_seq.append(y[i+SEQ_LEN])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y)

# =========================
# TIME-BASED SPLIT
# =========================
total = len(X_seq)

test_size = int(0.1 * total)
calib_size = int(0.1 * total)

train_size = total - test_size - calib_size

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]

X_calib = X_seq[train_size:train_size+calib_size]
y_calib = y_seq[train_size:train_size+calib_size]

X_test = X_seq[train_size+calib_size:]
y_test = y_seq[train_size+calib_size:]

# =========================
# DATASET
# =========================
class TSData(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TSData(X_train, y_train), batch_size=32, shuffle=True)

# =========================
# MODEL
# =========================
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 64, batch_first=True)
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)

        lower = out[:, 0]
        upper = lower + torch.abs(out[:, 1])  # ensure valid interval

        return lower, upper

model = LSTMModel(len(selected_features))

# =========================
# TUBE LOSS (Improved)
# =========================
def tube_loss(y, lower, upper, t=0.9, r=0.5, delta=0.01):
    loss = 0
    for i in range(len(y)):
        yi = y[i]
        l = lower[i]
        u = upper[i]

        if yi > u:
            loss += t * (yi - u)
        elif yi < l:
            loss += t * (l - yi)
        else:
            mid = r * u + (1 - r) * l
            if yi >= mid:
                loss += (1 - t) * (u - yi)
            else:
                loss += (1 - t) * (yi - l)

        loss += delta * torch.abs(u - l)

    return loss / len(y)

# =========================
# TRAIN
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        lower, upper = model(X_batch)
        loss = tube_loss(y_batch, lower, upper)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# =========================
# CONFORMAL CALIBRATION
# =========================
model.eval()

errors = []

with torch.no_grad():
    for i in range(len(X_calib)):
        x = torch.tensor(X_calib[i], dtype=torch.float32).unsqueeze(0)
        y_true = y_calib[i]

        l, u = model(x)
        l, u = l.item(), u.item()

        err = max(l - y_true, y_true - u)
        errors.append(err)

# 🔥 IMPORTANT: compute BEFORE test
alpha = 0.1
q_hat = np.quantile(errors, 1 - alpha)

print("Conformal Adjustment:", q_hat)
with open("lstm_errors.pkl", "wb") as f:
    pickle.dump(errors, f)

# =========================
# TEST EVALUATION
# =========================
inside = 0
outside = 0
results = []

with torch.no_grad():
    for i in range(len(X_test)):
        x = torch.tensor(X_test[i], dtype=torch.float32).unsqueeze(0)
        y_true = y_test[i]

        l, u = model(x)
        l, u = l.item(), u.item()

        l_adj = l - q_hat
        u_adj = u + q_hat

        if l_adj <= y_true <= u_adj:
            inside += 1
            status = "YES"
        else:
            outside += 1
            status = "NO"

        results.append([y_true, l_adj, u_adj, status])

# =========================
# METRICS
# =========================
total = inside + outside
coverage = inside / total

print("\n===== TEST RESULTS =====")
print(f"Total Points: {total}")
print(f"Inside Interval: {inside}")
print(f"Outside Interval: {outside}")
print(f"Coverage (PICP): {coverage:.4f}")

# =========================
# SAVE RESULTS
# =========================
df_results = pd.DataFrame(results, columns=["Actual", "Lower", "Upper", "Inside?"])
df_results.to_csv("interval_results.csv", index=False)

avg_width = np.mean(df_results["Upper"] - df_results["Lower"])
print("Average Width:", avg_width)

# =========================
# SAVE MODEL
# =========================
torch.save(model.state_dict(), "lstm_model.pth")

with open("qhat.pkl", "wb") as f:
    pickle.dump(q_hat, f)

print("Training complete ✅")

Selected Features: ['vol_15', 'vol_5', 'range', 'ret_15', 'lower_wick', 'dist_ma_5', 'dist_ma_20', 'upper_wick', 'bank_ret', 'log_ret']
Epoch 1, Loss: 0.7074
Epoch 2, Loss: 0.1234
Epoch 3, Loss: 0.1073
Epoch 4, Loss: 0.0984
Epoch 5, Loss: 0.0841
Epoch 6, Loss: 0.0748
Epoch 7, Loss: 0.0696
Epoch 8, Loss: 0.0664
Epoch 9, Loss: 0.0605
Epoch 10, Loss: 0.0628
Epoch 11, Loss: 0.0618
Epoch 12, Loss: 0.0670
Epoch 13, Loss: 0.0638
Epoch 14, Loss: 0.0587
Epoch 15, Loss: 0.0583
Epoch 16, Loss: 0.0586
Epoch 17, Loss: 0.0554
Epoch 18, Loss: 0.0568
Epoch 19, Loss: 0.0550
Epoch 20, Loss: 0.0692
Conformal Adjustment: -0.002791522660632726

===== TEST RESULTS =====
Total Points: 119
Inside Interval: 108
Outside Interval: 11
Coverage (PICP): 0.9076
Average Width: 0.038845839411357594
Training complete ✅


In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle

SEQ_LEN = 20

# =========================
# CONFIDENCE LEVELS 🔥
# =========================
CONFIDENCE_LIST = [0.90, 0.95, 0.99]

# =========================
# LOAD FILES
# =========================
with open("lstm_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("lstm_features.pkl", "rb") as f:
    features = pickle.load(f)

# 🔥 load calibration errors
with open("lstm_errors.pkl", "rb") as f:
    errors = pickle.load(f)

# =========================
# MODEL
# =========================
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 64, batch_first=True)
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)

        lower = out[:, 0]
        upper = lower + torch.abs(out[:, 1])

        return lower, upper

model = LSTMModel(len(features))
model.load_state_dict(torch.load("lstm_model.pth"))
model.eval()

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("nifty_features.csv")

X = df[features].values
X_latest = X[-SEQ_LEN:]
X_latest = scaler.transform(X_latest)

X_latest = torch.tensor(X_latest, dtype=torch.float32).unsqueeze(0)

# =========================
# BASE PREDICTION
# =========================
with torch.no_grad():
    lower, upper = model(X_latest)

lower = lower.item()
upper = upper.item()

print("\n===== MULTI-CONFIDENCE PREDICTION =====")

# =========================
# LOOP OVER CONFIDENCE 🔥
# =========================
for conf in CONFIDENCE_LIST:

    q_hat = np.quantile(errors, conf)

    lower_adj = lower - q_hat
    upper_adj = upper + q_hat

    pred = (lower_adj + upper_adj) / 2

    print(f"\nConfidence: {conf}")
    print(f"Lower: {lower_adj:.5f}")
    print(f"Upper: {upper_adj:.5f}")

    if pred > 0:
        print("Signal: BUY 📈")
    else:
        print("Signal: SELL 📉")


===== MULTI-CONFIDENCE PREDICTION =====

Confidence: 0.9
Lower: -0.02211
Upper: 0.03249
Signal: BUY 📈

Confidence: 0.95
Lower: -0.02656
Upper: 0.03693
Signal: BUY 📈

Confidence: 0.99
Lower: -0.05453
Upper: 0.06491
Signal: BUY 📈


C:\Users\Sahil Vasani\AppData\Local\Temp\ipykernel_17104\2086457418.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("lstm_model.pth"))
